# 07 — Ablaciones y selección (solo validación)

Implementación preparada el 2026-11-01 (America/Lima). **Etapa abierta hasta revisión de evidencia Colab de ambos regímenes.** Leer `docs/project/validacion_etapa7.md` y el cuaderno de decisiones antes de ejecutar.

Solo el usuario ejecuta modelos en Colab GPU. Sin entrenamiento, recalibración, test, Bolivia, Paraguay reservado ni demo. Las selecciones y locks se generan con mediciones reales; no existen resultados nuevos por crear este notebook.


## Hipótesis y controles previos

H1 WAIMLAP: referencia asistida > puntos DINOv3 > malla. H2: positivos mejoran hasta saturar; negativos reducen FP. H3: ajustar decoder ayuda más a eventos vistos que a Bolivia; **la parte geográfica no se contrasta aquí**.

A1: cuatro fuentes, decoder base congelado. Referencia asistida emparejada 3+/3− y distancia 16 px; se conserva además la referencia histórica 1+/1− de etapa 3. A2: k+ {1,3,5,10} con k−=3, después k− {0,1,3,5} con k+=3, recomendaciones registradas y sin reajuste de umbrales. A3: misma ruta directa de etapa 6 en ambas ramas, puntos/probe/umbrales idénticos, solo cambia decoder. No reutilizar automáticamente la ruta predictor como control de la ruta directa.

13 filas y 10 contrastes por régimen; 89/75 chips val. Todos los resultados, fallos y deltas negativos se conservan. NDWI aplazado/excluido. La selección automática usa IoU agua pooled, empate exacto por menor latencia desde imagen y luego ID estable. La calibración de etapa 5 y best de etapa 6 se registran aparte, sin nuevas búsquedas.


## 1. Entorno y paquete de fuentes

En PC ejecutar `scripts/validate_stage7_local.py` con el Python de `.venv-validation`; subir `artifacts/auto-prompt-sam2-stage7.zip` a MyDrive. Copiar su SHA en la siguiente celda. Mantener la raíz histórica completa con features y checkpoints de etapas 1–6; los ZIP de evidencia ligera no la reemplazan. Elegir runtime Colab con GPU. La instalación reconstruye el entorno fijado de Colab, no un entorno de modelos en PC.


In [5]:
from pathlib import Path
import os,sys,subprocess,zipfile,hashlib,json
PERSIST_ROOT='/content/drive/MyDrive/auto-prompt-sam2/auto-prompt-sam2'
REPO=Path('/content/auto-prompt-sam2-stage7')
BUNDLE=Path('/content/drive/MyDrive/auto-prompt-sam2/auto-prompt-sam2-stage7-reparado.zip')
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest()=='e9556df886da8bd6585f8c3a29558fdd0c2fea052a2ab96a74e60025dbe94638'
REPO.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z: z.extractall(REPO)
child_env=dict(os.environ,PYTHONPATH=str(REPO/'src'),AUTO_PROMPT_PERSIST_ROOT=PERSIST_ROOT,PYTHONHASHSEED='42',CUBLAS_WORKSPACE_CONFIG=':4096:8',AUTO_PROMPT_EXECUTION_CONTEXT='google_colab_user_notebook')
LOG_DIR=Path('/content/drive/MyDrive/auto-prompt-sam2/stage7/notebook_logs')
LOG_DIR.mkdir(parents=True,exist_ok=True)
def command(args,name):
    import time
    log_path=LOG_DIR/(str(time.time_ns())+'_'+name)
    print('LOG',log_path,flush=True)
    with log_path.open('x') as log:
        p=subprocess.Popen(args,cwd=REPO,env=child_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            log.write(line); log.flush(); print(line,end='')
        if p.wait(): raise RuntimeError('Proceso falló; revisar '+str(log_path))
command([sys.executable,'-m','pip','install','-q','uv'],'uv.log')
command([sys.executable,'-m','uv','python','install','3.12'],'python312.log')
py312=subprocess.check_output([sys.executable,'-m','uv','python','find','3.12'],text=True).strip()
PYTHON=Path('/content/auto-prompt-venv/bin/python')
command([py312,str(REPO/'scripts/bootstrap_colab.py')],'bootstrap_recover.log')
command([str(PYTHON),'-m','pip','install','--no-deps','-r',str(REPO/'requirements-stage2.txt')],'stage2_pins.log')
command([str(PYTHON),'-m','pip','check'],'pip_check.log')
print('READY_RECOVER')


LOG /content/drive/MyDrive/auto-prompt-sam2/stage7/notebook_logs/1795030908145021774_uv.log
LOG /content/drive/MyDrive/auto-prompt-sam2/stage7/notebook_logs/1795030912870469097_python312.log
Installed Python 3.12.16 in 1.76s
 + cpython-3.12.16-linux-x86_64-gnu (python3.12)
LOG /content/drive/MyDrive/auto-prompt-sam2/stage7/notebook_logs/1795030915060572674_bootstrap_recover.log
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.1 MB/s  0:00:00

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 110.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 27.4 MB/s  0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 19.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Revisar presupuesto cerrado

Ambos regímenes comparten espacio de búsqueda y controles antes de ver resultados nuevos. No cambiar esta matriz después de inspeccionar el régimen oficial. La diferencia de componentes y total real de puntos se reporta: igualdad de k por componente no garantiza igualdad de llamadas SAM.


In [ ]:
matrix = json.loads((REPO / 'configs/ablations.yaml').read_text('utf-8'))
print(json.dumps(matrix, ensure_ascii=False, indent=2))
print('13 filas x 89/75 chips val: 1157 + 975 = 2132 unidades lógicas, con cachés previas válidas reutilizadas.')
print('Perfil: primer chip val por régimen, 13 filas, 2 rutas cache, 5 warm-up + 20 repeticiones, batch una imagen.')
print('No hay entrenamiento ni nuevos candidatos fuera de esta matriz. GPU insuficiente = registrar pendiente.')


## 3. Ejecutar prepare, resume o recover

Primera ejecución: `RUN_MODE='prepare'`. Si hay interrupción técnica, detener el escritor anterior y usar `resume` con el mismo paquete, raíz y condiciones; las generaciones completas se reutilizan y los fallos publicados no se ocultan.

Después de completar ambos regímenes, **guardar este notebook con salidas prepare**, desconectar/eliminar runtime y abrir otro runtime GPU. Repetir solo entorno y esta sección con `RUN_MODE='recover'`. Reconstruye offline desde imagen el primer chip val en cada fila exitosa, verifica todas las generaciones y recalcula selección/locks sin alterarlos. Guardar también ese notebook con salidas. No ejecutar prepare dos veces.


In [6]:
RUN_MODE='recover'
command([str(PYTHON),str(REPO/'scripts/run_stage7.py'),RUN_MODE,'--root',PERSIST_ROOT],'recover_20261118.log')
print('RECOVERY_COMPLETE')


LOG /content/drive/MyDrive/auto-prompt-sam2/stage7/notebook_logs/1795031137917896819_recover_20261118.log
`torch_dtype` is deprecated! Use `dtype` instead!
official a1_points 1 / 89
official a1_points 21 / 89
official a1_points 41 / 89
official a1_points 61 / 89
official a1_points 81 / 89
official a1_points ok {'water_iou': 0.29861481371632737, 'precision': 0.32715268295226385, 'recall': 0.7739225645276981, 'dice': 0.45989743927494964, 'nonwater_iou': 0.7808819289476611, 'miou': 0.5397483713319942}
official a1_assisted 1 / 89
official a1_assisted 21 / 89
official a1_assisted 41 / 89
official a1_assisted 61 / 89
official a1_assisted 81 / 89
official a1_assisted ok {'water_iou': 0.15401975854556055, 'precision': 0.15502590446121636, 'recall': 0.9595652494519811, 'dice': 0.2669274202716866, 'nonwater_iou': 0.35013583341089183, 'miou': 0.2520777959782262}
official a1_whole_box 1 / 89
official a1_whole_box 21 / 89
official a1_whole_box 41 / 89
official a1_whole_box 61 / 89
official a1_whole

## 4. Ver resultados, exportar y detener

Se informa rendimiento por chip/evento, costo del prompt (también negativo), curvas A2, diferencias A3 y fallos. Los perfiles son del primer chip val, no estiman la latencia de toda la población; incluyen lectura SHA y las sincronizaciones de etapas, con cachés OS/Drive calientes. Se separa carga inicial y ejecución con/sin features; ambas recalculan embeddings SAM. SAM y DINO permanecen residentes, incluso cuando la rama no utiliza DINO.

Compartir notebook(s) con salidas, **dos ZIP terminados y SHA**, JSON selection/lock/validation/recovery/both_regimes y logs/failure. Los locks incluyen las 13 filas finales con hashes y `stage8_authorized=false` hasta revisión externa y cierre de ambos regímenes. No abrir reservados ni seguir con la demo.


In [7]:
import time
archives=[next((Path(PERSIST_ROOT)/'regimes'/r/'stage7').glob('*/evidence.zip')) for r in ['official','leave_paraguay_out']]
review_path=LOG_DIR.parent/('revision_stage7_20261118_'+str(time.time_ns())+'.json')
command([str(PYTHON),str(REPO/'scripts/review_stage7_evidence.py'),*[str(p) for p in archives],'--data-root',PERSIST_ROOT,'--output',str(review_path)],'audit_recover_20261118.log')
print('FINAL_REVIEW',review_path)
print('FINAL_REVIEW_SHA',hashlib.sha256(review_path.read_bytes()).hexdigest())


LOG /content/drive/MyDrive/auto-prompt-sam2/stage7/notebook_logs/1795033146261806003_audit_recover_20261118.log
{
  "pair": {
    "both_computational_locks_verified": true,
    "external_review": "pending",
    "stage8_authorized": false,
    "reserved_access": []
  },
  "regimes": {
    "official": {
      "status": "portable_consistency_passed_not_stage_closure",
      "zip_sha256": "df51ddc57ea3f9dac31443f1873d5fdcba8fadeb4f264d3245f7cc7c8a62c55b",
      "generation_count": 1252,
      "rows": 13,
      "selected_id": "a3_tuned",
      "failures": [],
      "validation_masks_independently_audited": 1157,
      "offline_recovery_present": true,
      "models_executed": false,
      "reserved_access": [],
      "pending": [
        "Human review of executed notebook, logs, hardware, failures and interpretation"
      ]
    },
    "leave_paraguay_out": {
      "status": "portable_consistency_passed_not_stage_closure",
      "zip_sha256": "89c24a9afb140e24a9335756ba5ef8d21b4fa2f0796d05b